# Run-level metadata extraction from file names

Extract metadata fields from proteomics run file names using regex pattern matching, then normalize to controlled vocabulary.

**Fields to extract:**
- organism, tissue, cell_line, cell_part, disease
- instrument, acquisition, fragmentation, labeling
- enrichment, fractionation

In [1]:
import pandas as pd
import re
from pathlib import Path

# Load the data

In [2]:
ROOT = Path(r"C:\Users\sander\OneDrive\Bureaublad\Projects\agentic-metadata")

dia_data_path = r"C:\Users\sander\OneDrive\Bureaublad\Projects\agentic-metadata\quant_data\diann_wide.parquet"
dda_data_path = r"C:\Users\sander\OneDrive\Bureaublad\Projects\agentic-metadata\quant_data\dnsaf_wide.parquet"
project_metadata_path = ROOT / "metadata.tsv"
ontology_dir = ROOT / "ontologies"

In [3]:
dia_runs = pd.read_parquet(dia_data_path, columns=["pxd", "run"])
dda_runs = pd.read_parquet(dda_data_path, columns=["pxd", "run"])
all_runs = pd.concat([dia_runs.assign(acquisition_type="DIA"), dda_runs.assign(acquisition_type="DDA")], ignore_index=True)
print(f"Total runs: {len(all_runs):,}  |  DIA: {len(dia_runs):,}  |  DDA: {len(dda_runs):,}")
all_runs.head()

Total runs: 89,626  |  DIA: 22,865  |  DDA: 66,761


,pxd,run,acquisition_type
0,PXD046357,20230919_OLEP08_HeLa_1cell_240k_20Th_40ms_1,DIA
1,PXD046357,20230919_OLEP08_HeLa_1cell_240k_20Th_40ms_10,DIA
2,PXD046357,20230919_OLEP08_HeLa_1cell_240k_20Th_40ms_11,DIA
3,PXD046357,20230919_OLEP08_HeLa_1cell_240k_20Th_40ms_12,DIA
4,PXD046357,20230919_OLEP08_HeLa_1cell_240k_20Th_40ms_2,DIA


In [4]:
def load_ontology(name: str) -> list[str]:
    """Load ontology terms from a text file, skipping comments and blanks."""
    terms = []
    for line in (ontology_dir / f"{name}.txt").read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            terms.append(line)
    return terms

# Build regex patterns per field

Strategy: for each field, build a mapping of `{regex_pattern: canonical_term}`.
Run names use `_`, `-`, or camelCase as delimiters, so word boundaries are `[_\-\s]` or start/end of string.

In [5]:
# Word boundary in run names: start/end of string, or common delimiters
B = r"(?:^|(?<=[\s_\-\.]))"    # look-behind boundary
B_FLEX = r"(?:^|(?<=[\s_\-\.\d]))"  # also allows digit before (for cell lines: 10HeLa)
A = r"(?=$|[\s_\-\.\d])"       # look-ahead boundary
A_CAMEL = r"(?=$|[\s_\-\.\d]|(?=[A-Z]))"  # also allows camelCase transition

def _b(pattern: str, canon: str, *, case=False, after=None, before=None) -> tuple[re.Pattern, object]:
    """Compile a bounded pattern -> canonical term."""
    flags = re.IGNORECASE if not case else 0
    head = before if before is not None else B
    tail = after if after is not None else A
    return (re.compile(head + pattern + tail, flags), canon)


# ---------- INSTRUMENT ----------
instrument_patterns = [
    # Orbitrap Exploris variants  (match numbered before generic)
    _b(r"(?:Orbitrap[\s_\-]?)?Exploris[\s_\-]?480", "Orbitrap Exploris 480"),
    _b(r"(?:Orbitrap[\s_\-]?)?Exploris[\s_\-]?240", "Orbitrap Exploris 240"),
    _b(r"(?:Orbitrap[\s_\-]?)?Exploris[\s_\-]?120", "Orbitrap Exploris 120"),
    _b(r"(?:Orbitrap[\s_\-]?)?Exploris", "Orbitrap Exploris 480"),  # generic Exploris
    _b(r"EXPL\d*", "Orbitrap Exploris 480"),           # EXPL1, EXPL8, EXPL11

    # Q Exactive variants  (match specific before generic)
    _b(r"Q[\s_\-]?Exactive[\s_\-]?HF[\s_\-]?X", "Q Exactive HF-X"),
    _b(r"QEHFX\d*", "Q Exactive HF-X"),
    _b(r"HFX\d*", "Q Exactive HF-X"),
    _b(r"qExHF\d*", "Q Exactive HF"),              # qExHF abbreviation
    _b(r"Q[\s_\-]?Exactive[\s_\-]?HF", "Q Exactive HF"),
    _b(r"QEHF\d*", "Q Exactive HF"),
    _b(r"qExPlus\d*", "Q Exactive Plus"),           # qExPlus02
    _b(r"Q[\s_\-]?Exactive[\s_\-]?Plus", "Q Exactive Plus"),
    _b(r"QEplus\d*", "Q Exactive Plus"),
    _b(r"QEp\d*", "Q Exactive Plus"),               # QEp2, QEp5, QEp7
    _b(r"qExac\d*", "Q Exactive"),                  # qExac abbreviation
    _b(r"Q[\s_\-]?Exactive", "Q Exactive"),
    _b(r"QE\d+", "Q Exactive"),                     # QE5, QE3 (require digit)
    _b(r"Qe\d+", "Q Exactive", case=True),          # Qe1

    # Orbitrap Eclipse
    _b(r"(?:Orbitrap[\s_\-]?)?Eclipse", "Orbitrap Eclipse"),

    # Orbitrap Fusion Lumos  (match before plain Fusion)
    _b(r"(?:Orbitrap[\s_\-]?)?Fusion[\s_\-]?Lumos", "Orbitrap Fusion Lumos"),
    _b(r"LUMOS\d*", "Orbitrap Fusion Lumos"),
    _b(r"(?:Orbitrap[\s_\-]?)?Fusion", "Orbitrap Fusion"),

    # Orbitrap Elite / Velos / XL
    _b(r"(?:LTQ[\s_\-]?)?Orbitrap[\s_\-]?Elite", "Orbitrap Elite"),
    _b(r"(?:LTQ[\s_\-]?)?Orbitrap[\s_\-]?Velos", "LTQ Orbitrap Velos"),
    _b(r"(?:LTQ[\s_\-]?)?Orbitrap[\s_\-]?XL", "LTQ Orbitrap XL"),
    _b(r"Orbitrap[\s_\-]?Astral", "Orbitrap Astral"),
    _b(r"Orbi\d+", "Orbitrap"),                     # Orbi2, Orbi3 (generic)

    # LTQ standalone
    _b(r"LTQ[\s_\-]?Velos", "LTQ Velos"),
    _b(r"Velos\d*", "LTQ Velos"),

    # timsTOF variants
    _b(r"timsTOF[\s_\-]?Ultra", "timsTOF Ultra"),
    _b(r"timsTOF[\s_\-]?SCP", "timsTOF SCP"),
    _b(r"timsTOF[\s_\-]?HT", "timsTOF HT"),
    _b(r"timsTOF[\s_\-]?Pro", "timsTOF Pro"),
    _b(r"timsTOF", "timsTOF Pro"),

    # SCIEX
    _b(r"TripleTOF[\s_\-]?6600", "TripleTOF 6600"),
    _b(r"TripleTOF[\s_\-]?5600", "TripleTOF 5600"),
    _b(r"ZenoTOF[\s_\-]?7600", "ZenoTOF 7600"),

    # Bruker
    _b(r"maXis", "maXis", case=True),
    _b(r"impact[\s_\-]?II", "impact II"),

    # Waters
    _b(r"Synapt", "Synapt G2-Si"),
]


# ---------- ACQUISITION ----------
# Must NOT be followed by a letter: "DDAlibrary" -> DDA, not missed
# DDA before DIA so DDA gets priority
_ACQ_AFTER = r"(?=$|[^A-Z])"
acquisition_patterns = [
    (re.compile(r"DDA" + _ACQ_AFTER), "DDA"),  # no left boundary, like TMT
    (re.compile(r"DIA" + _ACQ_AFTER), "DIA"),
]


# ---------- LABELING ----------
# TMT: no left boundary - appears in compounds like MnTMT18
labeling_patterns = [
    (re.compile(r"TMT"), "TMT"),  # case-sensitive, no boundaries
    _b(r"iTRAQ\d*(?:plex)?", "iTRAQ"),
    _b(r"SILAC", "SILAC", case=True),
]


# ---------- FRAGMENTATION ----------
fragmentation_patterns = [
    _b(r"EThcD", "EThcD"),
    _b(r"ETD", "ETD", case=True),
    _b(r"HCD", "HCD", case=True),
    _b(r"CID", "CID", case=True),
    _b(r"UVPD", "UVPD", case=True),
]


# ---------- ORGANISM ----------
# Use A_CAMEL so "MouseLiver" matches Mouse at the camelCase boundary
organism_patterns = [
    (re.compile(r"(?:Homo[\s_\-]?sapiens|[Hh]uman)", re.IGNORECASE), "Homo sapiens"),
    _b(r"(?:Mus[\s_\-]?musculus|[Mm]ouse)", "Mus musculus", after=A_CAMEL),
    _b(r"(?:Rattus[\s_\-]?norvegicus|[Rr]at)", "Rattus norvegicus", after=A_CAMEL),
    _b(r"(?:Danio[\s_\-]?rerio|[Zz]ebrafish)", "Danio rerio", after=A_CAMEL),
    _b(r"(?:Drosophila|[Ff]ly)", "Drosophila melanogaster", after=A_CAMEL),
    _b(r"(?:C[\.\s_]?elegans|[Ww]orm)", "Caenorhabditis elegans", after=A_CAMEL),
    _b(r"(?:S[\.\s_]?cerevisiae|[Yy]east)", "Saccharomyces cerevisiae", after=A_CAMEL),
    _b(r"(?:E[\.\s_]?coli)", "Escherichia coli", after=A_CAMEL),
    _b(r"(?:Arabidopsis)", "Arabidopsis thaliana", after=A_CAMEL),
]


# ---------- TISSUE ----------
_tissue_aliases = {
    # organs
    "brain": "brain", "heart": "heart", "liver": "liver", "lung": "lung",
    "kidney": "kidney", "pancreas": "pancreas", "spleen": "spleen",
    "stomach": "stomach", "colon": "colon", "rectum": "rectum",
    "esophagus": "esophagus", "duodenum": "duodenum",
    r"small[\s_\-]?intestine": "small intestine",
    "cecum": "cecum", "aorta": "aorta",
    "prostate": "prostate", "ovary": "ovary",
    r"testicul(?:ar)?": "testis", "testis": "testis",
    "uterus": "uterus", "cervix": "cervix",
    "retina": "retina", "skin": "skin", "breast": "breast",
    "gut": "gut", "tonsil": "tonsil", "placenta": "placenta",
    # body fluids
    "plasma": "blood plasma", "serum": "blood serum",
    "blood": "blood", "saliva": "saliva", "urine": "urine",
    "ascites": "ascites",
    "tear": "tear fluid", "milk": "milk", "sputum": "sputum",
    "feces": "feces", "stool": "feces", "sweat": "sweat",
    # tissue types
    "adipose": "adipose tissue", r"bone[\s_\-]?marrow": "bone marrow",
    r"lymph[\s_\-]?node": "lymph node", r"spinal[\s_\-]?cord": "spinal cord",
    r"frontal[\s_\-]?cortex": "frontal cortex", "cortex": "frontal cortex",
    
    # cell types
    "platelet": "platelets",
    "monocyte": "monocytes", "macrophage": "macrophage",
    "fibroblast": "fibroblast", "epithelial": "epithelial cell",
    "erythrocyte": "erythrocyte", "neutrophil": "neutrophil",
    "oocyte": "oocyte", "sperm": "sperm",
    # compound words in run names
    "MouseLiver": "liver", "MouseBrain": "brain", "MouseKidney": "kidney",
    "MouseHeart": "heart", "MouseLung": "lung",
    "SmallIntestine": "small intestine",
    "livernuc": "liver", "lowserum": "blood serum",
    "iPSCs?": "stem cells", "hiPSCs?": "stem cells", "hESCs?": "stem cells",
    r"Dentate[\s_\-]?Gyrus": "brain", r"Pyramidal[\s_\-]?Layer": "brain", r"Stratum[\s_\-]?Moleculare": "brain",
    "hippocampus": "brain", "cerebellum": "brain", "striatum": "brain", "hypothalamus": "brain",
    "amygdala": "brain", "thalamus": "brain", "midbrain": "brain",

}
tissue_patterns = []
for alias, canon in _tissue_aliases.items():
    if alias.isupper() and len(alias) <= 5:
        tissue_patterns.append(_b(alias, canon, case=True))
    else:
        tissue_patterns.append(_b(alias, canon))


# Tissue terms needing camelCase boundary (e.g. CardiacFibroblasts)
tissue_patterns.append(_b(r"cardiac", "heart", after=A_CAMEL))
tissue_patterns.append((re.compile(r"PBMCs?"), "PBMCs"))
tissue_patterns.append(_b(r"[Ss]erum", "blood serum", after=A_CAMEL))
tissue_patterns.append(_b(r"[Mm]yofibre", "skeletal muscle"))
tissue_patterns.append(_b(r"[Mm]yofiber", "skeletal muscle"))
tissue_patterns.append((re.compile(r"[Mm]uscle", re.IGNORECASE), "skeletal muscle"))
tissue_patterns.append((re.compile(r"[Ss]perm", re.IGNORECASE), "sperm"))
tissue_patterns.append((re.compile(r"[Hh]epatocyte", re.IGNORECASE), "liver"))


# ---------- CELL LINE ----------
cell_line_terms = load_ontology("cell_lines")

def _cell_line_pattern(term: str) -> tuple[re.Pattern, object]:
    """Build a regex that matches the cell line with optional delimiters.
    Uses B_FLEX (allows digit before) so 10HeLa matches."""
    escaped = re.escape(term)
    flexible = re.sub(r"\\[ \-]", r"[\\s_\\-]?", escaped)
    return (re.compile(B_FLEX + flexible + A, re.IGNORECASE), term)

cell_line_patterns = [_cell_line_pattern(t) for t in cell_line_terms]
# Common short aliases
cell_line_patterns.append(_b(r"HEK", "HEK293", before=B_FLEX))
cell_line_patterns.append(_b(r"Jur", "Jurkat", case=True, before=B_FLEX))


# ---------- CELL PART ----------
cell_part_patterns = [
    _b(r"mitochondria(?:l)?", "mitochondria"),
    _b(r"[Mm]ito", "mitochondria"),
    _b(r"nucle(?:us|ar|i)", "nucleus"),
    _b(r"Nuc", "nucleus", case=True),               # short form: _Nuc_
    _b(r"cytoplasm(?:ic)?", "cytoplasm"),
    _b(r"Cyt", "cytoplasm", case=True),              # short form: _Cyt_
    _b(r"membrane", "membrane"),
    _b(r"MEMB", "membrane", case=True),
    _b(r"ribosom(?:e|al|es)", "ribosome"),
    _b(r"lysosom(?:e|al|es)", "lysosome"),
    _b(r"peroxisom(?:e|al|es)", "peroxisome"),
]


# ---------- DISEASE ----------
disease_patterns = [
    # Healthy/control
    _b(r"[Hh]ealthy", "healthy"),
    _b(r"[Nn]on[\s_\-]?[Cc]ancerous", "healthy"),
    _b(r"[Bb]enign", "healthy"),
    _b(r"WT", "healthy", case=True),
    # Cancer terms
    _b(r"BrCa", "breast cancer", case=True),
    (re.compile(r"[Cc]ancer"), "cancer"),
    _b(r"[Tt]umor", "cancer"),
    _b(r"[Tt]umour", "cancer"),
    _b(r"[Cc]arcinoma", "cancer"),
    _b(r"[Mm]elanoma", "melanoma"),
    _b(r"[Gg]lioblastoma", "glioblastoma"),
    _b(r"[Ll]eukemia", "leukemia"),
    _b(r"[Ll]ymphoma", "lymphoma"),
    # Infectious
    _b(r"[Ii]nfluenza", "influenza"),
    _b(r"COVID", "COVID-19", case=True),
    _b(r"SARS", "COVID-19", case=True),
]


# ---------- ENRICHMENT ----------
# No left boundary for phospho/phos - distinctive enough, appears in compounds like DDAphos
enrichment_patterns = [
    (re.compile(r"[Pp]hosph", re.IGNORECASE), "phospho-enrichment"),  # also catches Phosphp typo
    _b(r"TiO\d", "phospho-enrichment", case=True),  # TiO1, TiO2 (elution fractions)
    (re.compile(r"[Pp]hos(?=[\s_\-\.])", re.IGNORECASE), "phospho-enrichment"),
    _b(r"ubiquit(?:in|yl)", "ubiquitin-enrichment"),
    _b(r"acetyl", "acetyl-enrichment"),
    _b(r"glyco(?:syl)?", "glyco-enrichment"),
    _b(r"SUMO", "SUMO-enrichment", case=True),
]


# ---------- FRACTIONATION ----------
# Only explicit fraction indicators. NOT: bRPLC (could be single-shot),
# SCX (ambiguous with SCIEX), FAIMS (ion mobility), gel (sample prep)
# Value is "True" string - just flags presence of fractionation
fractionation_patterns = [
    # Explicit fraction indicators (no left boundary - catches SECFraction etc.)
    (re.compile(r"[Ff]racc?(?:tion(?:at(?:ed|ion))?)?"), "True"),  # Frac, Fraction, Fracction, FractionA
    (re.compile(r"[Ff]r(?=[\dA-Z_])"), "True"),  # Fr14, FrD, fr_12
    (re.compile(r"[Ff]xn(?=\d)"), "True"),      # Fxn6
    (re.compile(r"(?<=[\s_\-\.])[fdFD]\d+(?=$|[\s_\.\-])"), "True"),  # _f11, _F10-2, _d3
]


# Collect all field -> pattern list mappings
FIELD_PATTERNS: dict[str, list[tuple[re.Pattern, object]]] = {
    "instrument": instrument_patterns,
    "acquisition": acquisition_patterns,
    "labeling": labeling_patterns,
    "fragmentation": fragmentation_patterns,
    "organism": organism_patterns,
    "tissue": tissue_patterns,
    "cell_line": cell_line_patterns,
    "cell_part": cell_part_patterns,
    "disease": disease_patterns,
    "enrichment": enrichment_patterns,
    "fractionation": fractionation_patterns,
}

print(f"Fields: {len(FIELD_PATTERNS)}")
for field, pats in FIELD_PATTERNS.items():
    print(f"  {field}: {len(pats)} patterns")

Fields: 11
  instrument: 41 patterns
  acquisition: 2 patterns
  labeling: 3 patterns
  fragmentation: 5 patterns
  organism: 9 patterns
  tissue: 83 patterns
  cell_line: 119 patterns
  cell_part: 11 patterns
  disease: 16 patterns
  enrichment: 7 patterns
  fractionation: 4 patterns


# Extract metadata from run names

In [ ]:
def extract_from_run_name(run_name: str) -> dict[str, str | None]:
    """Match all field patterns against a run name. Returns first match per field,
    except multi-valued fields (fragmentation) which collect all matches.
    For tissue: if both a body fluid and a solid tissue match, the fluid wins
    (e.g., 'human_plasma_liver_cancer' -> blood plasma, not liver)."""
    MULTI_FIELDS = {"fragmentation"}

    # Body fluids take priority over solid tissues when both match
    BODY_FLUIDS = {
        "blood plasma", "blood serum", "blood", "saliva", "urine",
        "ascites", "tear fluid", "milk", "sputum", "feces", "sweat",
        "cerebrospinal fluid", "seminal plasma",
    }

    result = {}
    for field, patterns in FIELD_PATTERNS.items():
        if field in MULTI_FIELDS:
            matches = []
            for regex, canon in patterns:
                if regex.search(run_name) and canon not in matches:
                    matches.append(canon)
            result[field] = "; ".join(str(v) for v in matches) if matches else None
        elif field == "tissue":
            # Collect ALL tissue matches, then apply body fluid priority
            all_matches = []
            for regex, canon in patterns:
                if regex.search(run_name) and canon not in all_matches:
                    all_matches.append(canon)
            if not all_matches:
                result[field] = None
            else:
                fluid_matches = [m for m in all_matches if m in BODY_FLUIDS]
                if fluid_matches:
                    result[field] = fluid_matches[0]
                else:
                    result[field] = all_matches[0]
        else:
            matched = None
            for regex, canon in patterns:
                if regex.search(run_name):
                    matched = canon
                    break
            result[field] = matched

    # Infer organism from cell line: all cell lines in our ontology are human
    if result.get("cell_line"):
        if result["organism"] is None:
            result["organism"] = "Homo sapiens"
        elif result["organism"] != "Homo sapiens":
            # Mixed sample (e.g. HeLa + Yeast) - keep both
            result["organism"] = result["organism"] + "; Homo sapiens"

    return result


# Run extraction on all runs
records = []
for _, row in all_runs.iterrows():
    extracted = extract_from_run_name(row["run"])
    extracted["pxd"] = row["pxd"]
    extracted["run"] = row["run"]
    records.append(extracted)

run_meta = pd.DataFrame(records)

# Reorder columns: identifiers first, then fields
id_cols = ["pxd", "run"]
field_cols = [c for c in run_meta.columns if c not in id_cols]
run_meta = run_meta[id_cols + sorted(field_cols, key=str)]

print(f"Extracted metadata for {len(run_meta):,} runs")
run_meta.head(10)


# Coverage statistics

In [7]:
print("Columns:", list(run_meta.columns))

print("=== Field coverage (% of runs with a match) ===")
for col in sorted(field_cols, key=str):
    n = run_meta[col].notna().sum()
    pct = n / len(run_meta) * 100
    print(f"  {str(col):<20s}  {n:>6,} / {len(run_meta):,}  ({pct:5.1f}%)")

# Show value distribution for key fields
print("=== Top values per field ===")
for col in sorted(field_cols, key=str):
    counts = run_meta[col].value_counts().head(10)
    if len(counts) > 0:
        print(f"--- {col} ---")
        for val, cnt in counts.items():
            print(f"  {str(val):<35s}  {cnt:>6,}")
        print()


Columns: ['pxd', 'run', 'acquisition', 'cell_line', 'cell_part', 'disease', 'enrichment', 'fractionation', 'fragmentation', 'instrument', 'labeling', 'organism', 'tissue']
=== Field coverage (% of runs with a match) ===
  acquisition           11,815 / 89,626  ( 13.2%)
  cell_line              8,506 / 89,626  (  9.5%)
  cell_part                284 / 89,626  (  0.3%)
  disease                1,281 / 89,626  (  1.4%)
  enrichment             2,219 / 89,626  (  2.5%)
  fractionation         15,941 / 89,626  ( 17.8%)
  fragmentation            406 / 89,626  (  0.5%)
  instrument            15,908 / 89,626  ( 17.7%)
  labeling               6,555 / 89,626  (  7.3%)
  organism              10,049 / 89,626  ( 11.2%)
  tissue                 4,894 / 89,626  (  5.5%)
=== Top values per field ===
--- acquisition ---
  DIA                                   9,912
  DDA                                   1,903

--- cell_line ---
  HeLa                                  3,779
  Jurkat                

# Spot-check: inspect matches for specific runs

In [8]:
# Spot-check: all user-reported cases
test_cases = [
    # (run_name, expected_fields_dict)
    ("B_0143_R_P11972_S127531_F01_MnTMT18_F06_R01",     {"labeling": "TMT"}),
    ("BrCa_cancertissue_T3",                              {"disease": "breast cancer"}),
    ("241206_06_DIA_livernuc_WT_CT10_2",                  {"tissue": "liver", "acquisition": "DIA"}),
    ("241229_02_DDAphos_liver_PER2SG_CT2_2",              {"enrichment": "phospho-enrichment", "acquisition": "DDA", "tissue": "liver"}),
    ("DIA_K10_aorta_powt",                                {"tissue": "aorta", "acquisition": "DIA"}),
    ("20231130_Liver_Nuc_total_CT02_01_DIA_24SPD_Iso2_3p5ms", {"tissue": "liver", "cell_part": "nucleus", "acquisition": "DIA"}),
    ("241018_01_DIA_Duodenum_CT2_1",                      {"tissue": "duodenum"}),
    ("241018_01_DIA_SmallIntestine_CT2_1",                {"tissue": "small intestine"}),
    ("241019_01_DIA_Cecum_CT2_1",                         {"tissue": "cecum"}),
    ("Palmitoylated_proteins_of_testicular_mitochondria_KO_rep1", {"tissue": "testis", "cell_part": "mitochondria"}),
    ("P4_Cyt_Tumor_1",                                    {"disease": "cancer", "cell_part": "cytoplasm"}),
    ("qExPlus02_01377Rep3",                               {"instrument": "Q Exactive Plus"}),
    ("20150414_QEp1_LC7_NiGr_SA_Saliva_1_fractionated1", {"tissue": "saliva", "fractionation": "True", "instrument": "Q Exactive Plus"}),
    ("14_RPE1-Resorption-72h-lowserum-1-5h10FCS-trypsin2", {"tissue": "blood serum"}),
    ("cachectic_melanoma_serum_1a",                        {"disease": "melanoma", "tissue": "blood serum"}),
    ("HF1_20160415_GMRJ1602B_TMT10_hpHRP_DMSO2_2p5ug_5uL_fr02", {"labeling": "TMT", "fractionation": "True"}),
    ("20160213_QE3_UPLC8_AKP_Jur_R3_Prot46F_35",         {"cell_line": "Jurkat", "instrument": "Q Exactive", "organism": "Homo sapiens"}),
    ("Ying_30um_10HeLa_25uLPrep_lowTryp_080217_1",       {"cell_line": "HeLa", "organism": "Homo sapiens"}),
    ("20180107_QX2_SeVW_SA_EASI6_HeLa_1-1-1-1-1-1_Yeast_1-3-10-10-3-1_fraction_7_1",
        {"cell_line": "HeLa", "organism": "Saccharomyces cerevisiae; Homo sapiens", "fractionation": "True"}),
    ("PreV-013_Fr14-15_R1",                               {"fractionation": "True"}),
    ("170616_IH_DIA_MIT_Lysosomes_F1",                    {"cell_part": "lysosome", "acquisition": "DIA"}),
    ("20180326_SA_Calu_Influenza_C_1_2",                  {"disease": "influenza"}),
    # Previous v2 cases
    ("OLEP07_200ng_CPR_HEK_180SPD_24_rep1_Frac16",       {"cell_line": "HEK293", "fractionation": "True", "organism": "Homo sapiens"}),
    ("Gastric_PP_DDAlibrary_AZ521",                       {"acquisition": "DDA"}),
    ("231020_FL_Secretome_M_plus_D_120min_Exploris_FAIMS_DIA_CV-45-60_R1", {"instrument": "Orbitrap Exploris 480", "acquisition": "DIA"}),
    ("In_gel_crude_sample_9",                             {}),  # no fractionation from gel
    ("PI027_SECFraction_EWZ_52_DIA",                      {"fractionation": "True", "acquisition": "DIA"}),
    ("BPRC_HF5_20200617_MouseLiver_QC_DIA_500ng_R1",     {"tissue": "liver", "organism": "Mus musculus", "acquisition": "DIA"}),
    ("20220304_Exploris_Project_569_proteome_1392",       {"instrument": "Orbitrap Exploris 480"}),
    ("HFX_10401_TFU_Control-P72-07_021219",              {"instrument": "Q Exactive HF-X", "disease": "healthy"}),
]

print("=== Spot-check results ===\n")
all_pass = True
for name, expected in test_cases:
    result = extract_from_run_name(name)
    hits = {k: v for k, v in result.items() if v is not None}
    # Check expected fields
    failures = []
    for field, exp_val in expected.items():
        got = result.get(field)
        if got != exp_val:
            failures.append(f"    {field}: expected '{exp_val}', got '{got}'")
    status = "PASS" if not failures else "FAIL"
    if failures:
        all_pass = False
    print(f"[{status}] {name}")
    print(f"  -> {hits}")
    for f in failures:
        print(f)
    print()

if all_pass:
    print("All spot-checks passed!")

=== Spot-check results ===

[PASS] B_0143_R_P11972_S127531_F01_MnTMT18_F06_R01
  -> {'labeling': 'TMT', 'fractionation': 'True'}

[PASS] BrCa_cancertissue_T3
  -> {'disease': 'breast cancer'}

[PASS] 241206_06_DIA_livernuc_WT_CT10_2
  -> {'acquisition': 'DIA', 'tissue': 'liver', 'disease': 'healthy'}

[PASS] 241229_02_DDAphos_liver_PER2SG_CT2_2
  -> {'acquisition': 'DDA', 'tissue': 'liver', 'enrichment': 'phospho-enrichment'}

[PASS] DIA_K10_aorta_powt
  -> {'acquisition': 'DIA', 'tissue': 'aorta'}

[PASS] 20231130_Liver_Nuc_total_CT02_01_DIA_24SPD_Iso2_3p5ms
  -> {'acquisition': 'DIA', 'tissue': 'liver', 'cell_part': 'nucleus'}

[PASS] 241018_01_DIA_Duodenum_CT2_1
  -> {'acquisition': 'DIA', 'tissue': 'duodenum'}

[PASS] 241018_01_DIA_SmallIntestine_CT2_1
  -> {'acquisition': 'DIA', 'tissue': 'small intestine'}

[PASS] 241019_01_DIA_Cecum_CT2_1
  -> {'acquisition': 'DIA', 'tissue': 'cecum'}

[PASS] Palmitoylated_proteins_of_testicular_mitochondria_KO_rep1
  -> {'tissue': 'testis', 'ce

# Export to TSV

In [9]:
out_path = r"C:\Users\sander\OneDrive\Bureaublad\Projects\agentic-metadata\notebooks\run_meta_name.tsv"
run_meta.to_csv(out_path, sep="\t", index=False)
print(f"Saved to {out_path}")
print(f"Shape: {run_meta.shape}")

Saved to C:\Users\sander\OneDrive\Bureaublad\Projects\agentic-metadata\notebooks\run_meta_name.tsv
Shape: (89626, 13)
